# Урок 18 — `@property`, `classmethod`, `staticmethod` та Дескриптори

**Модуль 2 · Python Intermediate · Автор: Viktor Nikoriak**

---

## Про що цей урок?

Сьогодні ми зазирнемо під капот об'єктної моделі Python.
Забудьте про синтаксис заради синтаксису. Ми поговоримо про те, **як Python мислить,  
як він будує об'єкти та як ви можете маніпулювати самим ядром цієї системи**.

| Частина | Тема | Ключова ідея |
|---------|------|--------------|
| 1 | `@property` | Контрольований доступ до внутрішнього стану |
| 2 | `classmethod` / `staticmethod` | Типи методів та патерн «Фабрика» |
| 3 | Дескриптори | Ядро механізму доступу до атрибутів |
| 4 | `__getattribute__` / `__getattr__` | Маршрутизація та пошук атрибутів |
| 5 | Метапрограмування | ORM, Pydantic, dataclasses під капотом |

> `property`, `classmethod`, `staticmethod` і навіть звичайні методи — це все
> «синтаксичний цукор» над **протоколом дескрипторів**.

---

## Частина 1: `@property` — Контрольований доступ до Стану

**Ментальна модель:** «Принцип єдиного доступу» (Uniform Access Principle).  
Клієнтський код не повинен знати, як саме реалізовано атрибут: чи це просте значення в пам'яті,
чи складне обчислення.

Для Java-програміста інкапсуляція означає написання `get_value()` та `set_value()`. Для Python-архітектора це **антипатерн**.  
У Python ми завжди починаємо з простих відкритих атрибутів (KISS-принцип), знаючи, що в будь-який
момент можемо перетворити їх на `property`, не зламавши публічний API.

---

### 1.1 Обчислювані атрибути (Computed Attributes)

Якщо стан об'єкта залежить від інших його даних — зберігати похідне значення окремо погана ідея
(доведеться постійно оновлювати вручну). Краще обчислювати **на льоту**.

---

**Передбачення:** Що зручніше — `c.area()` чи `c.area`? Що виведе код нижче?

In [ ]:
import math

class Circle:
    def __init__(self, radius):
        # Зберігаємо лише радіус — єдиний "справжній" стан об'єкта
        self.radius = radius

    @property
    def area(self):
        # Обчислюємо площу на льоту при кожному зверненні
        print("Обчислюємо площу...")
        return math.pi * self.radius ** 2

c = Circle(4.0)
print(c.radius)  # Звичайний атрибут
print(c.area)    # @property: звертаємось БЕЗ дужок, як до атрибута!

#c.area = 25  # Розкоментуйте -> AttributeError: can't set attribute


**Пояснення:**

`@property` перетворює метод на обчислюваний атрибут, доступний **тільки для читання**.
Замість суміші `c.radius` (атрибут) та `c.area()` (метод),
ми створюємо **уніфікований інтерфейс екземпляра**.

---

### 1.2 Валідація та контроль інваріантів

Справжня сила `@property` — захист **бізнес-правил (інваріантів)** від зовнішнього хаосу.
Наприклад, вага товару не може бути від'ємною.

---

**Передбачення:** Що станеться, якщо хтось спробує встановити `weight = 0`?

In [ ]:
class LineItem:
    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight   # Увага: тут викликається setter!
        self.price = price

    @property
    def weight(self):
        # Геттер: повертаємо приховану резервну змінну
        return self.__weight

    @weight.setter
    def weight(self, value):
        # Сеттер: "мембрана" між зовнішнім світом і внутрішнім станом
        if value > 0:
            self.__weight = value
        else:
            raise ValueError("Вага має бути > 0")

truffle = LineItem("White truffle", 100, 10.0)
print(truffle.weight)  # 100

# Ключова деталь: валідація спрацьовує навіть у __init__!
#truffle.weight = -5  # Розкоментуйте, щоб побачити захист у дії

**Пояснення:**

Зверніть увагу на геніальну деталь: в `__init__` ми пишемо `self.weight = weight` (не `self.__weight`).  
Це означає, що **наша валідація спрацьовує навіть під час створення об'єкта** — об'єкт ніколи
не потрапить у невалідний стан. Сеттер діє як надійна «мембрана».

---

### 1.3 Дебагінг: Пастка нескінченної рекурсії

Фактичні дані повинні зберігатись **поза межами самого `property`** — інакше виникне катастрофа.

**Завдання на дебагінг:** Цей код викликає `RecursionError`. Знайдіть причину та виправте.

> *Підказка: Звернення до `self.age` всередині методу `age` знову викликає той самий метод.*
> *Дані мають зберігатися у резервній (backing) змінній, наприклад, `self._age`.*

In [ ]:
# ЗЛАМАНИЙ КОД — демонструємо проблему
class BadPropertyExample:
    def __init__(self):
        self.age = 20        # Це викликає setter...

    @property
    def age(self):
        return self.age      # ПОМИЛКА: знову викликає age -> нескінченна рекурсія!

    @age.setter
    def age(self, value):
        self.age = value     # ПОМИЛКА: знову викликає setter -> нескінченна рекурсія!

# Розкоментуйте, щоб побачити RecursionError:
#person = BadPropertyExample()

In [ ]:
# ВИПРАВЛЕНИЙ КОД - зберігаємо дані у резервній змінній _age
class GoodPropertyExample:
    def __init__(self):
        # self.age викликає setter, який коректно збережає у self._age
        # self.age(20)  # викликає setter
        self.age = 20

    @property
    def age(self):
        # Читаємо зі збережної резервної змінної (одне підкреслення)
        return self._age

    @age.setter
    def age(self, value):
        if value < 0:
            raise ValueError("Вік не може бути від'ємним")
        # Зберігаємо у резервну змінну, а НЕ у self.age!
        self._age = value

person = GoodPropertyExample()
print(person.age)   # 20  # person.age()  # викликає getter
person.age = 25
print(person.age)   # 25

### 1.4 Архітектурні Антипатерни

**Антипатерн 1: Геттери і сеттери без логіки.**
```python
@property
def name(self): return self._name

@name.setter
def name(self, value): self._name = value
```
Це зайвий код без жодної архітектурної переваги над звичайним `self.name`.
Зробіть атрибут публічним. Якщо вимоги зміняться — ви легко обгорнете його `@property`.

**Антипатерн 2: Побічні ефекти в геттерах.**  
Метод з `@property` має поводитися як звичайне поле даних. Не робіть запити до бази даних
чи зміну стану інших об'єктів у геттері — це стане великим сюрпризом для розробників.

---

### 1.5 Практичне завдання: Клас `Temperature`

**Завдання:**
1. В `__init__` задайте початкову температуру в градусах Цельсія.
2. Створіть `@property` `celsius`, яке повертає значення.
3. Додайте `@celsius.setter`, який забороняє температуру нижче абсолютного нуля (-273.15).
4. Створіть обчислюване `@property` `fahrenheit` за формулою: `(C × 9/5) + 32`.

---

In [ ]:
class Temperature:
    def __init__(self, celsius):
        # Встановлюємо через сеттер - він одразу провалідує значення
        self.celsius = celsius

    @property
    def celsius(self):
        # Повертаємо резервну змінну
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        # Абсолютний нуль - мінімально допустима температура у Всесвіті
        if value < -273.15:
            raise ValueError("Температура не може бути нижче абсолютного нуля (-273.15 C)")
        self._celsius = value

    @celsius.deleter
    def celsius(self):
        print("Видаляємо температуру...")
        del self._celsius

    @property
    def fahrenheit(self):
        # Обчислюване поле: конвертуємо на льоту, нічого не зберігаємо
        return (self._celsius * 9 / 5) + 32


t = Temperature(0)
print(f"{t.celsius} C = {t.fahrenheit} F")   # 0 C = 32.0 F



t.celsius = 100
print(f"{t.celsius} C = {t.fahrenheit} F")   # 100 C = 212.0 F

try:
    t.celsius = -300
except ValueError as e:
    print(f"Помилка: {e}")

t = Temperature(25)

print(f"До видалення: {t.celsius}°C")

del t.celsius
print("Температуру видалено")

try:
    print(f"Після видалення: {t.celsius}°C")
except AttributeError:
    print("❌ Дані температури відсутні")

---

## Частина 2: `classmethod` та `staticmethod` — Типи методів

В Python тип зв'язування методу з його «одержувачем» визначає, який перший неявний аргумент він отримає:

| Тип методу | Перший аргумент | Доступ до стану |
|-----------|----------------|----------------|
| Instance method | `self` (екземпляр) | Стан екземпляра + класу |
| `@classmethod` | `cls` (клас) | Тільки стан класу |
| `@staticmethod` | — (нічого) | Не має доступу до стану |

---

### 2.1 Instance methods — стандарт

Стандартні методи приймають `self` — посилання на конкретний екземпляр.
Вони можуть читати та змінювати стан екземпляра, а також звертатись до класу через `self.__class__`.

---

### 2.2 `@classmethod` — Патерн «Фабрика» та Альтернативні конструктори

**Ментальна модель:** Класи в Python — це повноцінні об'єкти першого класу (екземпляри метакласу `type`).
`classmethod` — це операція, яка працює із самим об'єктом-шаблоном (класом).

**Чому не `__init__` з різними параметрами?**  
На відміну від Java/C++, Python не підтримує перевантаження методів.
Якщо вам потрібно створювати об'єкти з різних джерел даних — `classmethod` ваш інструмент.

**Чому не `staticmethod`?**  
`staticmethod` взагалі не отримує посилання на клас. Якщо ви створите підклас,
`staticmethod` не знатиме про це. `classmethod` завдяки `cls` завжди створює об'єкт
**саме того класу**, через який його було викликано (навіть якщо це нащадок).

---

In [ ]:
from datetime import datetime

class Point:
    def __init__(self, x, y):
        # Єдиний __init__ для базового випадку
        self.x = x
        self.y = y

    @classmethod
    def from_tuple(cls, coords):
        # Альтернативний конструктор: розпаковуємо кортеж
        # cls() замість Point() - працює коректно для підкласів!
        return cls(*coords)

    @classmethod
    def origin(cls):
        # Ще один альтернативний конструктор - точка на початку координат
        return cls(0, 0)

    @staticmethod
    def distance(p1, p2):
        # Утилітарна функція: не потребує ні self, ні cls
        # Групуємо у клас Point для логічного зв'язку
        return ((p1.x - p2.x)**2 + (p1.y - p2.y)**2) ** 0.5

    def __repr__(self):
        return f"Point({self.x}, {self.y})"


# Три способи створити Point
p1 = Point(3, 4)                 # Стандартний __init__
p2 = Point.from_tuple((1, 2))    # classmethod
p3 = Point.origin()              # classmethod

print(p1, p2, p3)
print(f"Відстань: {Point.distance(p1, p2):.2f}")

In [ ]:
# Переваги cls над hardcoded назвою класу - демонстрація на підкласі
class Point3D(Point):
    def __init__(self, x, y, z=0):
        super().__init__(x, y)
        self.z = z

    def __repr__(self):
        return f"Point3D({self.x}, {self.y}, {self.z})"


# from_tuple успадкований від Point, але cls = Point3D!
p3d = Point3D.from_tuple((5, 6))
print(type(p3d))   # <class '__main__.Point3D'> - правильний тип!
print(p3d)

# Якби в from_tuple стояло Point(*coords) замість cls(*coords) -
# тут би повернувся Point, а не Point3D. Це типова помилка!

### 2.3 `@staticmethod` — Утилітарні функції в просторі імен класу

Статичні методи отримують **ні `self`, ні `cls`**. Вони поводяться як звичайні функції,
але розміщені в просторі імен класу для логічного групування.

**Реальне використання:** Функції-помічники, які логічно відносяться до класу,
але не потребують доступу до стану об'єкта або класу.

```python
class TemperatureConverter:
    @staticmethod
    def celsius_to_fahrenheit(c):
        return (c * 9 / 5) + 32

    @staticmethod
    def fahrenheit_to_celsius(f):
        return (f - 32) * 5 / 9

# Можна викликати без екземпляра
print(TemperatureConverter.celsius_to_fahrenheit(100))  # 212.0
```

---

---

## Частина 3: Дескриптори — Ядро механізму атрибутів

**Ментальна модель:** Дескриптор — це об'єкт, який **бере на себе управління доступом**
до атрибута іншого класу.

Дескриптори — це базовий механізм для `@property`, `@classmethod`, `@staticmethod`,
`__slots__` і навіть звичайних методів класу.

---

### 3.1 Протокол дескриптора: `__get__`, `__set__`, `__delete__`

| Метод | Коли викликається | Аргументи |
|-------|------------------|-----------|
| `__get__(self, instance, owner)` | При читанні | `instance` — об'єкт (або `None` через клас) |
| `__set__(self, instance, value)` | При записі | `instance` — об'єкт; `value` — нове значення |
| `__delete__(self, instance)` | При `del obj.attr` | `instance` — об'єкт |

> **Критично:** У `__set__` значення слід зберігати в `instance.__dict__` або під іншим ім'ям.
> Зберігання на `self` (дескрипторі) — **помилка**: дескриптор є атрибутом **класу**, спільним для всіх екземплярів!

---

In [ ]:
# Базовий приклад власного дескриптора
class Verbose:
    # Дескриптор, який логує кожне звернення до атрибута

    def __set_name__(self, owner, name):
        # Автоматично отримуємо ім'я атрибута при визначенні класу
        self.public_name  = name
        self.private_name = "_" + name  # Резервна змінна у __dict__ екземпляра

    def __get__(self, instance, owner):
        if instance is None:
            # Доступ через клас: повертаємо сам дескриптор
            return self
        value = getattr(instance, self.private_name, "не задано")
        print(f"[GET] {self.public_name} -> {value!r}")
        return value

    def __set__(self, instance, value):
        print(f"[SET] {self.public_name} = {value!r}")
        # Зберігаємо у словник ЕКЗЕМПЛЯРА (не дескриптора!)
        setattr(instance, self.private_name, value)

    def __delete__(self, instance):
        print(f"[DEL] {self.public_name}")
        delattr(instance, self.private_name)


class Person:
    # Дескриптор як атрибут класу
    name = Verbose()
    age  = Verbose()

p = Person()
p.name = "Олена"   # -> __set__
p.age  = 30        # -> __set__
print(p.name)      # -> __get__
del p.age          # -> __delete__

### 3.2 Золоте правило: Data vs. Non-data дескриптори

| Тип | Реалізує | Пріоритет | Приклади |
|-----|----------|-----------|---------|
| **Data Descriptor** | `__set__` (і `__get__`) | Вищий: перекриває `instance.__dict__` | `property`, поля Django ORM |
| **Non-data Descriptor** | Тільки `__get__` | Нижчий: `instance.__dict__` перекриває його | Методи класу, `staticmethod` |

> **Висновок:** Ви ніколи не зможете «замаскувати» `property` через `obj.name = ...`,
> оскільки `property` — data-дескриптор (вищий пріоритет). Але ви можете перекрити звичайний
> метод класу присвоєнням `obj.my_method = 42` (метод — non-data дескриптор).

---

In [ ]:
# Демонстрація пріоритетів
class DataDescriptor:
    # Data дескриптор - реалізує __set__, тому має найвищий пріоритет
    def __get__(self, obj, objtype=None):
        return "значення з DATA-дескриптора"

    def __set__(self, obj, value):
        pass  # Ігноруємо запис для чистоти прикладу


class NonDataDescriptor:
    # Non-data дескриптор - тільки __get__, нижчий пріоритет
    def __get__(self, obj, objtype=None):
        return "значення з NON-DATA-дескриптора"


class MyClass:
    data_attr     = DataDescriptor()
    non_data_attr = NonDataDescriptor()


obj = MyClass()
print("data_attr     :", obj.data_attr)

print("non_data_attr :", obj.non_data_attr)
# Напряму вписуємо у словник екземпляра - спроба "перекрити" дескриптори
obj.__dict__["data_attr"]     = "значення з __dict__"
obj.__dict__["non_data_attr"] = "значення з __dict__"

# Data дескриптор ігнорує __dict__ - він завжди перемагає
print("data_attr     :", obj.data_attr)

# Non-data дескриптор програє __dict__ - __dict__ повертається першим
print("non_data_attr :", obj.non_data_attr)

### 3.3 Як `@property` влаштований зсередини

Вбудована функція `property` — це просто C-оптимізована реалізація дескриптора.
Ось її еквівалент на чистому Python:

---

In [ ]:
# Спрощена Python-реалізація вбудованого property
class Property:

    def __init__(self, fget=None, fset=None, fdel=None):
        self.fget = fget
        self.fset = fset
        self.fdel = fdel

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if self.fget is None:
            raise AttributeError("unreadable attribute")
        return self.fget(instance)

    def __set__(self, instance, value):
        if self.fset is None:
            raise AttributeError("can't set attribute")
        self.fset(instance, value)

    def __delete__(self, instance):
        if self.fdel is None:
            raise AttributeError("can't delete attribute")
        self.fdel(instance)

    def setter(self, fset):
        return Property(self.fget, fset, self.fdel)

    def deleter(self, fdel):
        return Property(self.fget, self.fset, fdel)


class Circle:
    def __init__(self, radius):
        self._radius = radius

    def _get_radius(self):
        return self._radius

    def _set_radius(self, v):
        if v <= 0:
            raise ValueError("Радіус має бути > 0")
        self._radius = v

    def _del_radius(self):
        print("Видаляємо радіус...")
        del self._radius

    radius = Property(_get_radius, _set_radius).deleter(_del_radius)


c = Circle(5)

print(c.radius)
c.radius = 10
print(c.radius)

del c.radius

try:
    print(c.radius)
except AttributeError:
    print("❌ Радіус видалено")

### 3.4 Практичні застосування дескрипторів

---

**A. Валідація атрибутів (ORM-стиль)**

Якщо у вас 50 атрибутів, які потребують перевірки типу — ви **не пишете 50 `property`**.
Ви пишете один клас-дескриптор і прив'язуєте його як атрибут класу.
Саме так влаштовані Django ORM, SQLAlchemy, Pydantic.

---

In [ ]:
class IntegerField:
    # Дескриптор-валідатор для цілих чисел (ORM-стиль)

    def __set_name__(self, owner, name):
        # Автоматично отримуємо ім'я атрибута при визначенні класу
        self.name         = name
        self.storage_name = "_field_" + name  # Унікальне ім'я у __dict__ екземпляра

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.storage_name, None)

    def __set__(self, instance, value):
        # Валідація типу - перехоплюємо будь-яке присвоєння
        if not isinstance(value, int):
            raise TypeError(
                f"Поле '{self.name}' має бути int, отримано {type(value).__name__!r}"
            )
        setattr(instance, self.storage_name, value)


class Product:
    # Один дескриптор замість кількох property
    quantity = IntegerField()
    stock    = IntegerField()

    def __init__(self, name, quantity, stock):
        self.name     = name
        self.quantity = quantity   # -> IntegerField.__set__
        self.stock    = stock      # -> IntegerField.__set__

p = Product("Яблуко", 50, 200)
print(p.quantity, p.stock)

try:
    p.quantity = 3.14   # Не ціле число!
except TypeError as e:
    print(f"Помилка: {e}")

---

**B. Ліниве кешування (Lazy Cached Properties)**

Ми використовуємо **слабкість non-data дескрипторів** собі на користь:
1. Перший доступ: дескриптор обчислює значення і записує результат в `instance.__dict__`.
2. Другий доступ: Python знаходить значення в `instance.__dict__` (вищий пріоритет)
   і повертає кеш, **повністю минаючи дескриптор**.

---

In [ ]:
class lazy_property:
    # Non-data дескриптор для одноразового обчислення і кешування

    def __init__(self, func):
        self.func = func
        self.name = None

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        # Обчислюємо значення один раз
        value = self.func(instance)
        # Записуємо у словник екземпляра - наступного разу дескриптор не викличеться!
        instance.__dict__[self.name] = value
        print(f"  [lazy_property] '{self.name}' обчислено i закешовано")
        return value


class Report:
    def __init__(self, data):
        self.data = data

    @lazy_property
    def summary(self):
        # Важке обчислення - виконується лише раз
        return {"count": len(self.data), "total": sum(self.data),
                "avg": sum(self.data) / len(self.data)}


r = Report([10, 20, 30, 40, 50])
print("Перший доступ:")
print(r.summary)   # Обчислюється i кешується

print("Другий доступ:")
print(r.summary)   # Повертається з __dict__ - без виклику дескриптора!

---

**C. Методи класу — це non-data дескриптори**

Всі user-defined функції є non-data дескрипторами, бо мають метод `__get__`.
Саме це і є механізмом «прив'язування» (binding) методів до екземпляра.

---

In [ ]:
class Greeter:
    def say_hello(self):
        return "Привіт!"

g = Greeter()

# Доступ до функції через клас - повертає звичайну функцію
print("Через клас:    ", type(Greeter.say_hello))   # <class 'function'>

# Доступ через екземпляр - функція стає "прив'язаним методом" (bound method)
print("Через екземпляр:", type(g.say_hello))         # <class 'method'>

# Функції мають __get__ - вони є дескрипторами!
print("Має __get__:", hasattr(Greeter.say_hello, "__get__"))  # True

# Ось як Python вручну "прив'язує" self - саме це робить __getattribute__ за лаштунками
bound = Greeter.say_hello.__get__(g, Greeter)
print("Прив'язаний виклик:", bound())

---

## Частина 4: Маршрутизація атрибутів — `__getattribute__` та `__getattr__`

Розуміння того, **як Python шукає `obj.attr`**, відрізняє Senior-розробника від мідла.

| | `__getattribute__` | `__getattr__` |
|---|---|---|
| **Коли викликається** | Безумовно при КОЖНОМУ зверненні | Тільки коли стандартний пошук **провалився** |
| **Призначення** | Абсолютний перехоплювач | Запасна сітка (fallback) |
| **Ризик рекурсії** | Дуже високий | Відсутній |
| **Правильний виклик** | `super().__getattribute__(name)` | — |
| **Застосування** | Логування, безпека | Патерн Проксі, делегування |

---

### 4.1 Ієрархія пошуку атрибутів (Алгоритм `__getattribute__`)

Коли `__getattribute__` шукає `obj.attr`, він використовує жорстку ієрархію:

```
obj.attr
   |
   +-- 1. Data Descriptor у class/MRO?    --> викликати __get__  (найвищий пріоритет)
   |
   +-- 2. instance.__dict__['attr']?      --> повернути значення
   |
   +-- 3. Non-data Descriptor у class?    --> викликати __get__
   |
   +-- 4. class.__dict__ / MRO?           --> повернути значення
   |
   +-- AttributeError  -->  __getattr__ (якщо визначений)
```

> **Мнемоніка:** Data -> Instance dict -> Non-data -> Class dict

---

In [ ]:
class SmartAccess:
    def __init__(self):
        self.name = "Python"

    def __getattribute__(self, name):
        # Перехоплює КОЖНЕ звернення - навіть до існуючих атрибутів
        print(f"[__getattribute__] -> \"{name}\"")
        # ОБОВ'ЯЗКОВО делегуємо до базового класу, щоб уникнути рекурсії!
        return super().__getattribute__(name)

    def __getattr__(self, name):
        # Викликається ТІЛЬКИ якщо __getattribute__ підняв AttributeError
        print(f"[__getattr__] \"{name}\" не знайдено -> динамічне значення")
        return f"<динамічне: {name}>"


obj = SmartAccess()
print("--- Існуючий атрибут ---")
print(obj.name)          # __getattribute__ -> знаходить у __dict__

print("--- Неіснуючий атрибут ---")
print(obj.unknown_attr)  # __getattribute__ -> AttributeError -> __getattr__

In [ ]:
# Практичний патерн: клас-Проксі на основі __getattr__
class Proxy:
    # Делегує всі невідомі звернення до внутрішнього об'єкта

    def __init__(self, target):
        # Зберігаємо через object.__setattr__ для уникнення можливих перехоплень
        object.__setattr__(self, "_target", target)

    def __getattr__(self, name):
        # Всі атрибути, яких немає у Proxy, делегуємо до _target
        target = object.__getattribute__(self, "_target")
        return getattr(target, name)


# Обгортаємо список у проксі
proxy = Proxy([1, 2, 3, 4, 5])

# Викликаємо методи списку через проксі
print(proxy.count(2))    # -> list.count, делеговано через __getattr__
print(len(proxy._target))  # 5

---

## Частина 5: Метапрограмування в реальних фреймворках

Метапрограмування — це **створення або налаштування класів динамічно під час виконання**.
Розуміння цього розкриває магію за Django ORM, SQLAlchemy, Pydantic, `dataclasses`.

---

### 5.1 ORM-поля як дескриптори (Django / SQLAlchemy)

Коли ви пишете:
```python
class Product(models.Model):
    price = models.DecimalField(max_digits=10, decimal_places=2)
```

`DecimalField(...)` — це **об'єкт-дескриптор**. Коли ви пишете `product.price = -5`,
фреймворк перехоплює присвоєння через `__set__`, виконує валідацію та відхиляє невалідні дані.

**Метакласи для реєстрації:** SQLAlchemy використовує декларативний метаклас.
Коли Python обробляє клас моделі, метаклас автоматично перехоплює його створення,
парсить поля-дескриптори, маппить їх до SQL-таблиці та реєструє клас у реєстрі ORM — 
без жодного boilerplate-коду з боку розробника.

---

### 5.2 `dataclasses` і `Pydantic` — метапрограмування на практиці

**`@dataclass`:** Декоратор `@dataclass` інспектує `__annotations__` (type hints) класу
і **автоматично генерує** методи `__init__`, `__repr__`, `__eq__` — ін'єктуючи їх у клас.

**Pydantic (runtime invariants):** Стандартні dataclasses не захищають інваріанти після ініціалізації —
можна легко обійти type hints прямим присвоєнням. Pydantic закриває цю прогалину:
він перехоплює **всі присвоєння полів** (через дескриптори) і виконує runtime-валідацію.

---

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Point:
    # @dataclass читає ці анотації та генерує __init__, __repr__, __eq__
    x: float
    y: float
    label: str = "точка"


p = Point(3.0, 4.0)
print(p)          # Автоматично згенерований __repr__
print(p == Point(3.0, 4.0))  # Автоматично згенерований __eq__

# Що насправді робить @dataclass - наближена демонстрація
print("\n--- Що генерує @dataclass ---")
import inspect
print(inspect.signature(Point.__init__))

In [ ]:
# Pydantic: runtime-валідація через дескриптори (потребує pip install pydantic)
# Цей блок демонструє концепцію - розкоментуйте якщо встановлено pydantic

# from pydantic import BaseModel, validator
#
# class Product(BaseModel):
#     name: str
#     price: float
#
#     @validator("price")
#     def price_must_be_positive(cls, v):
#         # Цей валідатор пов'язаний як дескриптор - спрацьовує при кожному присвоєнні
#         if v <= 0:
#             raise ValueError("Ціна має бути > 0")
#         return v
#
# p = Product(name="Яблуко", price=1.5)  # OK
# p = Product(name="Яблуко", price=-1)   # ValidationError!

# Демонстрація без pydantic: чому звичайний dataclass не захищає інваріанти
@dataclass
class ProductSimple:
    name: str
    price: float  # type hint, але НЕ runtime-валідація!

p = ProductSimple("Яблуко", 1.5)
p.price = -999   # Dataclass дозволяє це! Немає захисту після __init__
print(f"price = {p.price}")  # -999 - інваріант порушено

---

## Підсумок уроку

### 1. `@property` — Контрольований доступ
- Починайте з **публічних атрибутів**; переходьте на `property` лише при потребі в логіці.
- `@property` + `@name.setter` = «мембрана» навколо внутрішнього стану.
- Зберігайте дані у **резервній змінній** (`_name`) для уникнення рекурсії.

### 2. `classmethod` / `staticmethod`
- `@classmethod`: операція над самим класом; патерн «Фабрика» / альтернативні конструктори;
  `cls` замість hardcoded імені класу — правильно працює з підкласами.
- `@staticmethod`: утилітарна функція в просторі імен класу; не отримує ні `self`, ні `cls`.

### 3. Дескриптори — Ядро механізму
- **Data Descriptor** (`__get__` + `__set__`): вищий пріоритет, ніж `instance.__dict__`.
- **Non-data Descriptor** (тільки `__get__`): `instance.__dict__` перекриває його.
- Один дескриптор замінює N `property` — ідеально для повторюваної валідаційної логіки.

### 4. Маршрутизація атрибутів
- `__getattribute__` — **абсолютний перехоплювач**; всередині завжди `super().__getattribute__()`.
- `__getattr__` — **безпечний fallback**; ідеальний для патерну Проксі та делегування.
- Порядок: **Data Desc -> `__dict__` -> Non-data Desc -> Class dict -> `__getattr__`**.

### 5. Метапрограмування
- ORM-поля — дескриптори; метакласи — автоматична реєстрація при створенні класу.
- `@dataclass` генерує методи з `__annotations__`; Pydantic захищає інваріанти через runtime-validation.

---

> *«Descriptor protocol is the magic that makes Python's object system work.»*
> — Luciano Ramalho, Fluent Python


*Протокол дескрипторів — це магія, яка змушує об’єктну систему Python працювати.*
___
👉 Descriptor protocol = це набір методів:
- __get__
- __set__
- __delete__
___
👉 які Python використовує, щоб вирішити:
- як читати атрибут
- як записувати атрибут
- як видаляти атрибут